# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohamed-Al-Saudi/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I chose **Lane 2: Refresh / Content Opportunity Scoring**. This lane helps content teams decide which pages to refresh first instead of checking all manually.

My lane is SCORING + RANKING (Learning to Rank).

Why not classification or clustering? Classification gives yes/no refresh, but we need ORDERED list. Clustering groups similar pages, but we need priority.

So: Input = page features (search_volume, avg_position, days_since_last_update, impressions_last_30d vs prev_30d, ctr). Output = refresh_priority_score 0-100, then sorted descending to get ranked queue P0/P1/P2.

**One row = One content_id** that could be refreshed.

In [1]:
task_type = "scoring + ranking - learning to rank"
print(task_type)

scoring + ranking - learning to rank


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Ideal target: Expected traffic gain (impressions/clicks) if we refresh this content now vs doing nothing, measured after 60 days. We cannot know this before acting.

**Proxy target I will predict now:** refresh_priority_score 0-100 built from REAL columns in content_refresh_anonymized.csv:

- traffic_decay_score from impressions_last_30d vs impressions_prev_30d + trend_pct
- content_age_score from days_since_last_update + content_age_days
- keyword_opportunity_score from search_volume * avg_position (5-20 striking distance) + competition
- engagement_score from low ctr + low engagement_rate

**Formula** v1: 0.4*traffic_decay + 0.3*keyword_opportunity + 0.2*content_age + 0.1*engagement, normalized 0-100.

**Label source:** Defined rule proxy for now. Later ML model trained on historical refreshes will predict real uplift and replace weights. Proxy is explainable: "P0 because -45% impressions + search_volume 5400 at position 8 + 400 days since update".

In [5]:
proxy_target = "refresh_priority_score 0-100"
ideal_target = "impressions_uplift_60d after refresh"
print(f"Proxy: {proxy_target}")
print(f"Ideal: {ideal_target}")

Proxy: refresh_priority_score 0-100
Ideal: impressions_uplift_60d after refresh


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Offline metric:** `NDCG@50`. Measures if truly best pages are in top 50 of my ranking. 1.0 = perfect ranking. Good >0.85. Why NDCG not accuracy? Because order matters - mistake at rank 1 costs more than rank 490.

**Secondary:** Spearman rank correlation >0.7 vs expert ranking on 100-page sample. `Precision@20`: Of top 20 P0, how many expert also says P0.

**Online business metric:** Time-to-queue. Manual triage of 500 pages in spreadsheet = ~3 hours (to validate with 1 ops interview) vs my system = 5 minutes measured with stopwatch + Loom video. Good = <10 min for 500 pages AND expert says "top 20 looks right".

**Action supported:** SEO Ops uses top 20 P0 to assign sprint this week, P1 next sprint, P2 backlog.

In [6]:
offline_metric = "NDCG@50 > 0.85"
online_metric = "Time-to-queue: 3h manual -> 5min system"
print(f"Offline: {offline_metric}")
print(f"Online: {online_metric}")

Offline: NDCG@50 > 0.85
Online: Time-to-queue: 3h manual -> 5min system


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = One content_id / one page** that could be refreshed.

**Dataset:**../../data/raw/content_refresh_anonymized.csv - 45 columns.

**Each row contains** search data (search_volume, avg_position, ctr, competition) + engagement (impressions_90d, clicks_90d, sessions_90d, engagement_rate) + age (content_age_days, days_since_last_update, freshness_tier).

**Target column** `refresh_priority_score` will be derived from these real columns and shows priority to refresh.

In [8]:
import pandas as pd
import numpy as np

# Load Lane 2 real dataset
df = pd.read_csv(r"/content/content_refresh_anonymized.csv")
print(f"Unit: ONE ROW = ONE content_id")
print(f"Shape: {df.shape} rows, {df.shape[1]} columns")
display(df.head(10))

Unit: ONE ROW = ONE content_id
Shape: (30000, 44) rows, 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,15000-25000,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,15000-25000,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,NaN,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,15000-25000,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,NaN,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2


In [9]:
# Build proxy target from REAL columns
df["traffic_decay_pct"] = ((df["impressions_prev_30d"] - df["impressions_last_30d"]) / (df["impressions_prev_30d"]+1) * 100).fillna(0)
df["traffic_decay_score"] = np.clip(df["traffic_decay_pct"], 0, 100)

df["content_age_score"] = np.clip(df["days_since_last_update"] / 365 * 100, 0, 100)

df["keyword_opportunity_score"] = np.where(
    (df["avg_position"] >= 5) & (df["avg_position"] <= 20),
    np.clip(np.log1p(df["search_volume"]) / np.log1p(df["search_volume"].max()) * 100, 0, 100),
    np.clip(np.log1p(df["search_volume"]) / np.log1p(df["search_volume"].max()) * 30, 0, 30)
)

df["engagement_score"] = (1 - df["ctr"].fillna(0))*20 + (1 - df["engagement_rate"].fillna(0))*20

df["refresh_priority_score"] = (
    0.4*df["traffic_decay_score"] +
    0.3*df["keyword_opportunity_score"] +
    0.2*df["content_age_score"] +
    0.1*df["engagement_score"]
).round(1).clip(0,100)

df["priority_label"] = pd.cut(df["refresh_priority_score"], bins=[0,40,70,100], labels=["P2","P1","P0"])

df_ranked = df.sort_values("refresh_priority_score", ascending=False).reset_index(drop=True)

print("\n--- Target column created ---")
display(df_ranked[["content_id","refresh_priority_score","priority_label","traffic_decay_pct","avg_position","search_volume","days_since_last_update","ctr","trend_pct"]].head(10))
print(df["priority_label"].value_counts())


--- Target column created ---


,content_id,refresh_priority_score,priority_label,traffic_decay_pct,avg_position,search_volume,days_since_last_update,ctr,trend_pct
0,content_bbca724138f2,75.9,P0,98.113208,12.1,1600.0,236,0.00,-100.0
1,content_3f7dbbd55f0c,74.4,P0,97.500000,10.2,14800.0,104,0.00,-98.3
2,content_02b0d6e30129,71.8,P0,95.000000,6.9,110.0,313,0.00,-95.6
3,content_e7eb94e121b9,69.9,P1,83.617747,15.6,22200.0,104,0.00,-83.9
4,content_f421fb2e10ea,68.6,P1,89.705882,5.9,5400.0,104,0.00,-91.0
5,content_12e48d4b449d,68.4,P1,90.000000,17.0,27100.0,20,0.00,-100.0
6,content_e8fc703f7ef0,68.0,P1,96.507115,9.5,9900.0,14,0.00,-96.5
7,content_03452bf379ce,67.7,P1,95.652174,7.6,1600.0,104,0.00,-100.0
8,content_71b773ffcf59,65.6,P1,96.551724,7.9,1300.0,104,0.97,-100.0
9,content_8ba781dafa55,65.3,P1,85.567434,9.0,2900.0,104,0.00,-85.6


priority_label
P2    22280
P1     2765
P0        3
Name: count, dtype: int64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Fixed rule fails:**

1. Weights arbitrary: Why 0.4 for traffic decay? Why not 0.35? Fixed rule = my gut feeling. ML learns optimal weights from historical refreshes that actually drove uplift in impressions/clicks.

2. Non-linear interactions: 50% traffic drop at avg_position 8 with search_volume 10k and competition low is more urgent than same drop at position 25 volume 100 high competition. Fixed if/else becomes spaghetti: if decay>40 and pos<10 and vol>1000 and age>300... ML tree model captures these interactions automatically from data.

3. Thresholds break: What is old? 180 days for news, 600 days for evergreen content. What is high volume? Depends on client. Fixed thresholds break across content_type and main_intent. ML adapts per tier if we feed age_tier, impression_tier, position_tier as features.

4. Scale: With 5000 pages and 45 features, manual rule tuning impossible. ML retrains weekly as new impressions data comes.

**Core first, AI second:** Core = My scoring logic and ranking framework + Ops workflow (P0 refresh this week). AI second = Use ML to optimize weights instead of hardcoding 0.4/0.3/0.2/0.1. I can explain with SHAP why page got P0 using real features avg_position, search_volume, days_since_last_update. Without ML queue works but brittle. With ML it improves over time and weights are data-driven defensible on call.

In [10]:
reason = "ML beats rule: learned weights from data vs arbitrary, captures non-linear interactions, adapts thresholds per tier, scales to 5k pages"
print(reason)

ML beats rule: learned weights from data vs arbitrary, captures non-linear interactions, adapts thresholds per tier, scales to 5k pages


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.